In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from statistics import mean
from sklearn.metrics import accuracy_score
import sys
from urllib.request import urlopen

sys.path.insert(0, "../")
# from model_training.preprocessor_research_paper import preprocess_data
from preprocessor import preprocess_data

In [ ]:
NUM_ROWS = 10

In [ ]:
df = pd.read_csv(
    "../data/uci_phishing_url_dataset.csv",
    index_col=False,
)
# randomise data
# df = df.sample(frac=1, ignore_index=True)
df

In [ ]:
res = df["URL"]

## include only urls that are alive

In [ ]:
# for idx, row in df["URL"].items():
#     # check url status
#     print(f"Doing {idx}, URL: {row}")
#     # f.write(f"Doing {idx}, URL: {row}")
#     try:
#         res = urlopen(row, timeout=4)
#         if res.getcode() < 400:
#             print("url is alive!")
#         else:
#             raise ValueError("Url failed")
#     except Exception as e:
#         # f.write(e)
#         print(e)
#         continue
#     pass


# TIMEOUT_EXTRACT_DATA = 4
# num_url_failed = 0
# stored_url: List[str] = []
# # f = open("log_url_status.txt", "a+", encoding="latin-1")
# new_pd: pd.DataFrame = pd.DataFrame()
# for idx, row in df.iterrows():

#     url = row["URL"]
#     # check url status
#     # print(f"Doing {idx}, URL: {row}")
#     # f.write(f"Doing {idx}, URL: {row}")
#     try:
#         res = urlopen(url, timeout=TIMEOUT_EXTRACT_DATA)
#         if res.getcode() < 400:
#             # print("url is alive!")
#             pd.concat([new_pd, row], ignore_index=True)
#         else:
#             raise ValueError("Url failed")
#     except Exception as e:
#         # f.write(e)
#         # print(e)
#         continue
#     pass

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from typing import List
import numpy as np

LOG_PATH = "log_url_status.txt"
counter_lock = Lock()
scan_counter = {"count": 0}


def log_progress(count: int) -> None:
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"Scanned {count} URLs\n")


def test_url_alive(df_chunk: pd.DataFrame) -> pd.DataFrame:
    TIMEOUT_EXTRACT_DATA = 4
    valid_rows: List[dict] = []

    for _, row in df_chunk.iterrows():
        url = row["URL"]
        try:
            res = urlopen(url, timeout=TIMEOUT_EXTRACT_DATA)
            if res.getcode() < 400:
                # Store the whole row as a dict so columns are preserved
                valid_rows.append(row.to_dict())
            else:
                raise ValueError("Url failed")
        except Exception:
            continue
        finally:
            with counter_lock:
                scan_counter["count"] += 1
                if scan_counter["count"] % 1000 == 0:
                    log_progress(scan_counter["count"])

    # Return a DataFrame with the same columns as input
    if valid_rows:
        return pd.DataFrame(valid_rows, columns=df_chunk.columns)
    return pd.DataFrame(columns=df_chunk.columns)


# Split the dataframe into 10 chunks safely using pandas iloc
chunk_size = int(np.ceil(len(df) / 10))
df_chunks = [df.iloc[i : i + chunk_size] for i in range(0, len(df), chunk_size)]

NUM_THREADS = 10
result_dfs = []

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = [executor.submit(test_url_alive, chunk) for chunk in df_chunks]
    for future in as_completed(futures):
        res_df = future.result()
        if not res_df.empty:
            result_dfs.append(res_df)

# Combine all valid rows and write with headers
if result_dfs:
    new_pd = pd.concat(result_dfs, ignore_index=True)
else:
    new_pd = pd.DataFrame(columns=df.columns)

new_pd.to_csv("cleaned_data.csv", index=False)

## Rename the column `Label` to is `IsLegit`

In [ ]:
df.rename(columns={"Label": "IsLegit"}, inplace=True)
df.to_csv("../data/url_label_test_data.csv", index=False)
df

## Obtain a subset of the columns

In [ ]:
select_rows = df.iloc[:NUM_ROWS]
select_rows

# Run the preprocessor and get the data into dataframe

In [ ]:
for idx, row_data in select_rows.iterrows():
    out_row_data = preprocess_data("https://" + row_data["URL"]).get_data()
    print(out_row_data)
    # CANNOT USE PREPROCESS DATA AS NOT ASYNC USE NORMAL PYTHON FILE